# Tutorial 9: Embedding Benchmark

How much information does a perturbation embedding carry for a
specific prediction task? This tutorial shows how to answer that
question using `embpy.tl.benchmark_embeddings`.

The idea is simple:

1. You have an `AnnData` with perturbation identifiers and a target
   you want to predict (e.g. IC50, gene expression).
2. You provide (or generate) embeddings for those perturbations.
3. `benchmark_embeddings` trains standard ML regressors on the
   embeddings and reports how well they predict the target.

If a model achieves high R2/Pearson, the embedding **encodes
information relevant to that task**. If all models score poorly, the
embedding may be missing the biology that matters.

### Supported regressors

| Name | sklearn class | Notes |
|------|--------------|-------|
| `"linear"` | `LinearRegression` | Baseline: purely linear mapping |
| `"ridge"` | `Ridge` | Regularised linear model (better with many dimensions) |
| `"knn"` | `KNeighborsRegressor` | Non-parametric: uses local neighborhood |
| `"random_forest"` | `RandomForestRegressor` | Non-linear ensemble |

### Metrics

| Metric | Meaning |
|--------|---------|
| **MSE** | Mean squared error (lower is better) |
| **R2** | Coefficient of determination (1 = perfect, 0 = random, negative = worse than mean) |
| **Pearson** | Pearson correlation between predicted and actual |
| **Spearman** | Spearman rank correlation (captures monotonic relationships) |
| **Delta L2 MAE** | Mean abs error of perturbation effect magnitude (expression only, requires controls) |
| **Delta L2 Pearson** | Correlation of predicted vs actual perturbation magnitude |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from anndata import AnnData

import embpy.tl as tl

---
## 1. Setup: Create Synthetic Data

We simulate a perturbation dataset with:
- 80 drug perturbations + 20 DMSO controls
- A scalar target (`ic50`) in `.obs`
- An expression matrix in `.X` (100 genes)
- Two pre-computed embeddings of different dimensionality

We make **embedding A** slightly correlated with IC50 (informative)
and **embedding B** purely random (uninformative) so we can see
a clear difference in benchmark scores.

In [ ]:
rng = np.random.default_rng(42)
n_pert, n_ctrl, n_genes = 80, 20, 100
n_total = n_pert + n_ctrl

# Latent factor that drives both IC50 and expression
latent = rng.standard_normal((n_pert, 8))

# IC50 is a noisy linear function of the latent factor
ic50_pert = latent @ rng.standard_normal(8) + rng.normal(0, 0.5, n_pert)
ic50_ctrl = np.full(n_ctrl, np.nan)  # controls have no IC50

# Expression: perturbation effect is latent-driven; controls are baseline
expr_pert = latent @ rng.standard_normal((8, n_genes)) + rng.normal(0, 0.3, (n_pert, n_genes))
expr_ctrl = rng.normal(0, 0.1, (n_ctrl, n_genes))  # near-zero baseline

X_expr = np.vstack([expr_pert, expr_ctrl]).astype(np.float32)

# Embedding A: includes the latent factor + noise -> informative
emb_a = np.vstack([
    np.hstack([latent, rng.standard_normal((n_pert, 56))]),
    rng.standard_normal((n_ctrl, 64)),
]).astype(np.float32)

# Embedding B: pure random noise -> uninformative
emb_b = rng.standard_normal((n_total, 128)).astype(np.float32)

obs = pd.DataFrame(
    {
        "drug_name": [f"drug_{i}" for i in range(n_pert)] + [f"DMSO_{i}" for i in range(n_ctrl)],
        "condition": ["perturbation"] * n_pert + ["DMSO"] * n_ctrl,
        "ic50": np.concatenate([ic50_pert, ic50_ctrl]),
    },
    index=[str(i) for i in range(n_total)],
)

adata = AnnData(X=X_expr, obs=obs)
adata.obsm["X_informative"] = emb_a
adata.obsm["X_random"] = emb_b

print(f"AnnData: {adata.n_obs} obs x {adata.n_vars} vars")
print(f"Conditions: {adata.obs['condition'].value_counts().to_dict()}")
print(f"Embeddings: {list(adata.obsm.keys())}")

---
## 2. Scalar Target: Predicting IC50

The simplest benchmark: predict IC50 from the embedding.
We use only the perturbation samples (controls have no IC50).

**What to look for:** If R2 > 0 and Pearson is substantial,
the embedding captures information about drug potency.

In [ ]:
# Only use perturbation samples for IC50 prediction
adata_pert = adata[adata.obs["condition"] == "perturbation"].copy()

results_a = tl.benchmark_embeddings(
    adata_pert,
    perturbation_column="drug_name",
    perturbation_type="chemical",
    target="ic50",
    obsm_key="X_informative",
)

print("=== Informative Embedding (X_informative) ===")
print(results_a[["mse", "r2", "pearson", "spearman"]].round(4))

In [ ]:
results_b = tl.benchmark_embeddings(
    adata_pert,
    perturbation_column="drug_name",
    perturbation_type="chemical",
    target="ic50",
    obsm_key="X_random",
)

print("=== Random Embedding (X_random) ===")
print(results_b[["mse", "r2", "pearson", "spearman"]].round(4))

As expected, the informative embedding (which contains the latent
factor driving IC50) performs substantially better across all models
and metrics. The random embedding scores near zero or negative R2,
confirming it has no useful signal for this task.

---
## 3. Expression Target with Controls

Now we predict the full expression profile (`.X`) and provide
controls so that the **delta L2** metric can be computed.

Delta L2 measures how well the model predicts the *magnitude* of the
perturbation effect relative to the control baseline:

- `delta_actual = ||expression_perturbed - mean_control||`
- `delta_predicted = ||expression_predicted - mean_control||`
- **Delta L2 MAE**: `mean(|delta_predicted - delta_actual|)` (lower is better)
- **Delta L2 Pearson**: correlation between predicted and actual magnitudes (higher is better)

In [ ]:
results_expr = tl.benchmark_embeddings(
    adata,
    perturbation_column="drug_name",
    perturbation_type="chemical",
    target="X",
    obsm_key="X_informative",
    control_column="condition",
    control_value="DMSO",
)

print("=== Expression prediction (informative embedding, with controls) ===")
print(results_expr.round(4))

In [ ]:
# Compare with random embedding
results_expr_rand = tl.benchmark_embeddings(
    adata,
    perturbation_column="drug_name",
    perturbation_type="chemical",
    target="X",
    obsm_key="X_random",
    control_column="condition",
    control_value="DMSO",
)

print("=== Expression prediction (random embedding, with controls) ===")
print(results_expr_rand.round(4))

The informative embedding achieves:
- Higher R2 and Pearson — it can reconstruct expression profiles.
- Lower Delta L2 MAE — it better predicts how far the perturbation
  moved expression from the control baseline.
- Higher Delta L2 Pearson — it correctly ranks which perturbations
  have stronger effects.

---
## 4. Comparing Embeddings Side-by-Side

A common use case: you have generated embeddings with two different
models (e.g. ChemBERTa vs MolFormer) and want to know which one is
more useful for your task. Simply run the benchmark twice and
compare.

In [ ]:
comparison = pd.concat(
    {
        "informative": results_a[["r2", "pearson"]],
        "random": results_b[["r2", "pearson"]],
    },
    axis=1,
)
print("=== IC50 prediction: Informative vs Random ===")
print(comparison.round(4))

---
## 5. Effect of Dimensionality Reduction

High-dimensional embeddings (512-D, 1024-D) can overfit with small
datasets. The `reduce_dim` parameter applies PCA before training.

This lets you test whether compressing the embedding helps or hurts.

In [ ]:
results_pca = tl.benchmark_embeddings(
    adata_pert,
    perturbation_column="drug_name",
    perturbation_type="chemical",
    target="ic50",
    obsm_key="X_informative",
    reduce_dim=8,
)

print("=== With PCA to 8 dims ===")
print(results_pca[["mse", "r2", "pearson"]].round(4))
print()
print("=== Without PCA (64 dims) ===")
print(results_a[["mse", "r2", "pearson"]].round(4))

In this synthetic example, PCA to 8 dimensions may improve linear
models (less noise to fit) but could slightly hurt tree-based models
(which handle high dimensions well). In real data, trying a few
values (8, 16, 32, 50) is a good practice.

---
## 6. Selecting Specific Models

If you only care about linear models (e.g. for interpretability), you
can select a subset of regressors.

In [ ]:
results_linear = tl.benchmark_embeddings(
    adata_pert,
    perturbation_column="drug_name",
    perturbation_type="chemical",
    target="ic50",
    obsm_key="X_informative",
    models=["linear", "ridge"],
)

print(results_linear[["r2", "pearson"]].round(4))

---
## 7. Rigorous Mode: 5-Fold CV + Hyper-Parameter Search

The quick mode is great for fast exploration. But when you need
**statistically solid numbers** (e.g. for a paper or a final model
comparison), use `mode="rigorous"`:

1. **Randomised grid search** finds the best hyper-parameters for each
   regressor using 5-fold CV optimising MSE.
2. A second **5-fold evaluation** with the best hyper-parameters computes
   all metrics (MSE, R2, Pearson, Spearman, delta L2) per fold.
3. Results are reported as **mean +/- std** across folds.

This is slower but gives you confidence intervals and tuned models.

In [ ]:
results_rigorous = tl.benchmark_embeddings(
    adata_pert,
    perturbation_column="drug_name",
    perturbation_type="chemical",
    target="ic50",
    obsm_key="X_informative",
    models=["ridge", "knn", "random_forest"],
    mode="rigorous",
)

print("=== Rigorous Mode (5-fold CV) ===")
print(results_rigorous[["r2_mean", "r2_std", "pearson_mean", "pearson_std"]].round(4))
print()
print("Best hyper-parameters:")
for model, row in results_rigorous.iterrows():
    print(f"  {model}: {row['best_params']}")

The `_std` columns tell you how stable each metric is across folds.
A high mean R2 with a low std means the result is reliable.
A high std suggests the model's performance depends heavily on which
data ends up in the test fold.

---
## 8. Plotting Benchmark Results

`embpy.pl.plot_benchmark` produces a grouped bar chart from the results
DataFrame. It auto-detects whether you ran quick or rigorous mode:

- **Quick mode**: plain bars.
- **Rigorous mode**: bars with error bars showing +/- 1 standard deviation.

`embpy.pl.plot_benchmark_comparison` goes a step further and compares
multiple embeddings side-by-side.

In [ ]:
import embpy.pl as pl

# Quick mode plot — no error bars
fig_quick = pl.plot_benchmark(results_a, title="Quick Mode — IC50")
fig_quick

In [ ]:
# Rigorous mode plot — bars with error bars (std from 5-fold CV)
fig_rigorous = pl.plot_benchmark(results_rigorous, title="Rigorous Mode — IC50 (5-fold CV)")
fig_rigorous

### Comparing Embeddings

When you have benchmarked multiple embeddings, pass them as a
dictionary to `plot_benchmark_comparison`:

In [ ]:
# Run rigorous mode on the random embedding for comparison
results_random_rigorous = tl.benchmark_embeddings(
    adata_pert,
    perturbation_column="drug_name",
    perturbation_type="chemical",
    target="ic50",
    obsm_key="X_random",
    models=["ridge", "knn", "random_forest"],
    mode="rigorous",
)

# Side-by-side comparison with error bars
fig_cmp = pl.plot_benchmark_comparison(
    {"Informative": results_rigorous, "Random": results_random_rigorous},
    metrics=["r2", "pearson", "spearman"],
)
fig_cmp

The error bars make it immediately clear whether the difference between
embeddings is statistically meaningful or within noise range.

| Mode | Use case | Speed |
|------|----------|-------|
| `"quick"` | Exploration, prototyping | Fast (single split) |
| `"rigorous"` | Papers, final evaluation | Slower (5-fold CV + grid search) |

---
## Summary

| Parameter | Description |
|-----------|-------------|
| `perturbation_column` | `.obs` column with perturbation IDs (or `"obs_names"`) |
| `perturbation_type` | `"genetic"` or `"chemical"` |
| `target` | `"X"` for expression, or `.obs` column name (e.g. `"ic50"`) |
| `obsm_key` | Pre-computed embedding key in `.obsm` |
| `embedding_model` | Model name to generate embeddings on the fly |
| `control_column` / `control_value` | Identify control observations for delta L2 |
| `models` | Which regressors to use |
| `mode` | `"quick"` (single split) or `"rigorous"` (5-fold CV + grid search) |
| `reduce_dim` | Apply PCA before benchmarking |
| `test_size` | Train/test split ratio — only in quick mode (default 0.2) |

**Workflow:**

```python
import embpy.tl as tl
import embpy.pl as pl

# Quick exploration
results = tl.benchmark_embeddings(
    adata, perturbation_column="drug_name",
    perturbation_type="chemical",
    target="ic50", obsm_key="X_chemberta",
)
pl.plot_benchmark(results)

# Rigorous evaluation (papers, final comparison)
results = tl.benchmark_embeddings(
    adata, perturbation_column="gene_name",
    perturbation_type="genetic",
    target="X", embedding_model="esm2_650M",
    control_column="condition", control_value="non-targeting",
    mode="rigorous",
)
pl.plot_benchmark(results)  # shows error bars from 5-fold CV

# Compare multiple embeddings
pl.plot_benchmark_comparison({"ChemBERTa": res_a, "MolFormer": res_b})
```

The results DataFrame and all intermediate data (embeddings, PCA) are
stored back into the AnnData for downstream analysis with `embpy.pl`.